# Fine-Tuning Reasoning LLMs with Group Relative Policy Optimization (GRPO)

## Overview

This notebook demonstrates how to fine-tune a reasoning-capable Large Language Model using **Group Relative Policy Optimization (GRPO)**, a reinforcement learning algorithm designed to improve multi-step reasoning without requiring a separate value model.

The implementation uses the Hugging Face ecosystem, including **TRL**, **Transformers**, **Datasets**, and **PEFT**, to train a compact language model with custom reward functions.

The workflow follows the same principles used in modern reasoning models such as **DeepSeek R1**.

In [ ]:
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off
!pip install -qqq flash-attn --no-build-isolation --progress-bar off

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
  Preparing metadata (setup.py) ... done
  ERROR: Failed building wheel for flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install -U wandb

In [ ]:
!pip install -q \
datasets==3.2.0 \
transformers==4.47.1 \
trl==0.14.0 \
peft==0.14.0 \
accelerate==1.2.1 \
bitsandbytes==0.45.2

# Training Workflow

The GRPO training pipeline consists of the following stages:

1. Load a reasoning dataset.
2. Initialize a pretrained language model.
3. Define reward functions that evaluate generated responses.
4. Configure the GRPO training algorithm.
5. Fine-tune the model using reinforcement learning.
6. Evaluate the resulting reasoning model.

## Dataset

The notebook uses a reasoning dataset consisting of prompts and expected responses. During training, the model generates multiple candidate solutions for each prompt, which are evaluated using custom reward functions rather than supervised labels.

In [ ]:
!pip install -U wandb

  Using cached wandb-0.28.0-py3-none-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached wandb-0.28.0-py3-none-manylinux_2_28_x86_64.whl (26.8 MB)
  Attempting uninstall: wandb
    Found existing installation: wandb 0.19.7
    Uninstalling wandb-0.19.7:
      Successfully uninstalled wandb-0.19.7


In [ ]:
import torch
import wandb
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# Log to Weights & Biases
wandb.login()

# Load dataset
dataset = load_dataset("mlabonne/smoltldr")
print(dataset)

wandb: WARNING Unable to verify login in offline mode.


DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})


## Loading the Base Model

A pretrained language model serves as the initialization for reinforcement learning. Instead of training from scratch, GRPO improves the model's reasoning capabilities through reward-driven optimization.

In [ ]:
# Load model
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load LoRA
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

ERROR:bitsandbytes.cextension:Could not load bitsandbytes native library: /lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.32' not found (required by /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cpu.so)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 85, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 72, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039
None


## Reward Functions

<p align="center">
<img src="Reward_F.jpg" width="650">
</p>

Reward functions evaluate the quality of generated responses and provide the learning signal for reinforcement learning. Multiple reward functions can be combined to encourage correctness, proper formatting, helpful reasoning, and other desired behaviors.

In [ ]:
# Reward function
def reward_len(completions, **kwargs):
    return [-abs(50 - len(completion)) for completion in completions]

## Group Relative Policy Optimization (GRPO)

<p align="center">
<img src="GRPO.jpg" width="850">
</p>

Unlike traditional supervised learning, GRPO generates multiple responses for the same prompt and evaluates them as a group. Responses receiving higher relative rewards are reinforced during optimization, allowing the model to progressively improve its reasoning strategies.

In [8]:
# Training arguments
training_args = GRPOConfig(
    output_dir="GRPO",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_prompt_length=512,
    max_completion_length=96,
    num_generations=4,
    #optim="adamw_8bit",
    optim="adamw_torch_fused",
    num_train_epochs=1,
    bf16=True,
    report_to=["wandb"],
    remove_unused_columns=False,
    logging_steps=1,
)

# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_len],
    args=training_args,
    train_dataset=dataset["train"],
)

# Train model
wandb.init(project="GRPO")
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
1,0.000000
2,0.000100
3,0.000100
4,0.000100
5,0.000100
6,0.000100
7,0.000000
8,0.000100
9,0.000100
10,0.000100


Step,Training Loss
1,0.000000
2,0.000100
3,0.000100
4,0.000100
5,0.000100
6,0.000100
7,0.000000
8,0.000100
9,0.000100
10,0.000100


TrainOutput(global_step=500, training_loss=0.01511306264437735, metrics={'train_runtime': 6745.4171, 'train_samples_per_second': 0.296, 'train_steps_per_second': 0.074, 'total_flos': 0.0, 'train_loss': 0.01511306264437735})

## Push Model to Hub

In [14]:
# Save model
merged_model = trainer.model.merge_and_unload()
merged_model.push_to_hub(
    "Miladsaeedi70/smollm-135m-instruct-grpoV2",
    private=False)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ba05340/model.safetensors:  18%|#7        | 48.0MB /  269MB            

CommitInfo(commit_url='https://huggingface.co/Miladsaeedi70/smollm-135m-instruct-grpoV2/commit/2c9388c241c7e9c4f915511915caa33607368609', commit_message='Upload LlamaForCausalLM', commit_description='', oid='2c9388c241c7e9c4f915511915caa33607368609', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Miladsaeedi70/smollm-135m-instruct-grpoV2', endpoint='https://huggingface.co', repo_type='model', repo_id='Miladsaeedi70/smollm-135m-instruct-grpoV2'), pr_revision=None, pr_num=None)

In [16]:
tokenizer.push_to_hub("Miladsaeedi70/smollm-135m-instruct-grpoV2")

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Miladsaeedi70/smollm-135m-instruct-grpoV2/commit/20b21ef50dc955cbf8e0b15c446d8115afd816db', commit_message='Upload tokenizer', commit_description='', oid='20b21ef50dc955cbf8e0b15c446d8115afd816db', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Miladsaeedi70/smollm-135m-instruct-grpoV2', endpoint='https://huggingface.co', repo_type='model', repo_id='Miladsaeedi70/smollm-135m-instruct-grpoV2'), pr_revision=None, pr_num=None)

## Inference

After training, the fine-tuned model is evaluated by generating responses for unseen prompts. Comparing these outputs with those of the base model provides insight into how reinforcement learning improves reasoning quality.

## Generate Text

In [ ]:
prompt = """
# A long document about the Cat

The cat (Felis catus), also referred to as the domestic cat or house cat, is a small
domesticated carnivorous mammal. It is the only domesticated species of the family Felidae.
Advances in archaeology and genetics have shown that the domestication of the cat occurred
in the Near East around 7500 BC. It is commonly kept as a pet and farm cat, but also ranges
freely as a feral cat avoiding human contact. It is valued by humans for companionship and
its ability to kill vermin. Its retractable claws are adapted to killing small prey species
such as mice and rats. It has a strong, flexible body, quick reflexes, and sharp teeth,
and its night vision and sense of smell are well developed. It is a social species,
but a solitary hunter and a crepuscular predator. Cat communication includes
vocalizations—including meowing, purring, trilling, hissing, growling, and grunting—as
well as body language. It can hear sounds too faint or too high in frequency for human ears,
such as those made by small mammals. It secretes and perceives pheromones.
"""

messages = [
    {"role": "user", "content": prompt},
]

In [18]:
# Generate text
from transformers import pipeline

generator = pipeline("text-generation", model="Miladsaeedi70/smollm-135m-instruct-grpoV2")

## Or use the model and tokenizer we defined earlier
# generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

generate_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.5,
    "min_p": 0.1,
}

generated_text = generator(
    messages,
    **generate_kwargs
)

print(generated_text)

Device set to use cuda:0


[{'generated_text': [{'role': 'user', 'content': '\n# A long document about the Cat\n\nThe cat (Felis catus), also referred to as the domestic cat or house cat, is a small\ndomesticated carnivorous mammal. It is the only domesticated species of the family Felidae.\nAdvances in archaeology and genetics have shown that the domestication of the cat occurred\nin the Near East around 7500 BC. It is commonly kept as a pet and farm cat, but also ranges\nfreely as a feral cat avoiding human contact. It is valued by humans for companionship and\nits ability to kill vermin. Its retractable claws are adapted to killing small prey species\nsuch as mice and rats. It has a strong, flexible body, quick reflexes, and sharp teeth,\nand its night vision and sense of smell are well developed. It is a social species,\nbut a solitary hunter and a crepuscular predator. Cat communication includes\nvocalizations—including meowing, purring, trilling, hissing, growling, and grunting—as\nwell as body language. I

# Results

The GRPO fine-tuning pipeline successfully demonstrates how reinforcement learning can improve the reasoning behavior of a pretrained language model.

### Key Components

- Pretrained language model
- LoRA parameter-efficient fine-tuning
- Custom reward functions
- GRPO optimization
- Reinforcement learning with TRL
- Hugging Face Hub integration

### Learning Outcomes

- Fine-tuned an LLM using reinforcement learning.
- Designed custom reward functions for reasoning tasks.
- Applied parameter-efficient adaptation using LoRA.
- Generated responses using a GRPO-trained reasoning model.

# Conclusion

This notebook demonstrated the complete workflow for fine-tuning a reasoning language model using Group Relative Policy Optimization (GRPO). By combining pretrained language models, custom reward functions, and reinforcement learning, GRPO enables efficient post-training that improves reasoning capabilities without requiring a separate value model. These techniques form an important part of modern reasoning-oriented LLM development.